# Triplet information decomposition — counts → pairs → triplets

How much of each co-proximity rung is genuinely new versus a restatement of the rung below it?

1. **pairs → triplets** — given the z of the three constituent pairs (AB, AC, BC), how well is the
   triplet z (ABC) determined? `1 − R²` = the irreducible 3-way signal (added value of triplets).
2. **counts → pairs** — given marker abundance, how much of the pair z do we already know? (The z is
   already degree/abundance-corrected, so this is a *residual* question; counts trivially predict the
   raw count `J` via the null mean `EW`.)

Two views of the pair→triplet redundancy: **empirical** (leave-cells-out CV R², within/between-cell)
and **mechanistic** (a "no-3-way-interaction" pairwise prediction of the wedge → the *connected* z).
Both the **selected** 171 candidate triplets and an unbiased **random** baseline are analysed.

Self-contained (like `chunglu_triplets_analysis.ipynb`): **Step 0** builds the combined triplet list and
submits/tracks the per-cell wedge LSF job from inside the notebook; the analysis cells read its outputs.
Run top-to-bottom on a compute kernel; re-run the Step 0 cell until 15/15 samples are done, then continue.

Data: `results/chunglu/infodecomp/*_triplets_percell.parquet` (per-center `W_c*`/`EW_c*` columns from the
extended `chunglu_triplets.py`) + `results/chunglu/*_doublets.parquet` + the annotated adata.

In [ ]:
import sys, json
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen")
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import polars as pl
import info_decomp_utils as idu
from info_decomp_utils import split_triplet, display_name

sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (5, 3), "axes.titlesize": 11, "font.family": "sans-serif"})

BASE = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
CACHE = BASE / "cache"
RESULTS = BASE / "results"
OUT = RESULTS / "chunglu"
INFO = OUT / "infodecomp"
ANNOTATED = CACHE / "adata_cytovi_annotated_compat.h5ad"
SAMPLES = ["S001", "S002", "S003", "S004", "S005", "S006", "S007", "S008",
           "S009", "S010", "S011", "S012", "S013", "S014", "S016"]

def tlab(m1, m2, m3):
    return "·".join(display_name(x) for x in (m1, m2, m3))

## Step 0 — build the combined list + submit/track the per-cell wedge LSF job

Heavy per-cell compute (671 triplets × 15 samples) runs on LSF, writing per-center-augmented parquets to
`results/chunglu/infodecomp/`. This cell builds the combined selected+random list if missing, submits the
array job if outputs are absent and nothing is queued, and reports progress. **Re-run it to refresh** until
15/15 samples are done, then continue to the analysis below. (Same pattern as `chunglu_triplets_analysis.ipynb`.)

In [ ]:
(RESULTS / "logs").mkdir(parents=True, exist_ok=True)
INFO.mkdir(parents=True, exist_ok=True)
LIST_JSON = OUT / "triplet_infodecomp_list.json"
if not LIST_JSON.exists():
    print("building combined selected+random list ->", LIST_JSON.name)
    get_ipython().system(f"cd {BASE} && {sys.executable} build_random_triplet_list.py")
n_trip = len(json.loads(LIST_JSON.read_text())["triples"])

_jobs = get_ipython().getoutput("bjobs -w 2>/dev/null")
qjobs = [l for l in _jobs if "clid" in l.lower()]
done = sorted(p.name.split("_")[0] for p in INFO.glob("*_triplets_percell.parquet"))

if qjobs:
    print(f"clid jobs already in queue ({len(qjobs)}):"); print("\n".join(qjobs))
elif len(done) >= len(SAMPLES):
    print(f"all {len(done)}/{len(SAMPLES)} per-cell outputs present — nothing to submit.")
else:
    print(f"submitting per-cell array: {n_trip} triplets × {len(SAMPLES)} samples ...")
    get_ipython().system(f"cd {BASE} && bsub < run_chunglu_triplets_infodecomp.lsf")

print(f"\n{len(done)}/{len(SAMPLES)} samples done:", done)
miss = [s for s in SAMPLES if s not in done]
print("waiting on:", miss if miss else "none — complete")
get_ipython().system(f"tail -n 3 {RESULTS}/logs/clid_*.err 2>/dev/null | tail -n 12")

## Load — triplet + doublet per-cell z and build the joined table

In [ ]:
adata = sc.read_h5ad(ANNOTATED)
meta = adata.obs[["cell_type_annot", "cell_system", "condition", "time"]].copy()
print("adata:", adata.shape)

cl = json.loads((OUT / "triplet_infodecomp_list.json").read_text())
sets_map = {tuple(sorted(n)): s for n, s in zip(cl["names"], cl["sets"])}
print("list sets:", pd.Series(cl["sets"]).value_counts().to_dict())

tfiles = sorted(INFO.glob("*_triplets_percell.parquet"))
assert tfiles, "no per-cell outputs yet — run the Step 0 submit/track cell and wait for the LSF job (re-run until 15/15)."
trp = pd.concat([pd.read_parquet(p) for p in tfiles], ignore_index=True)
assert "EW_cA" in trp.columns, "old schema — rerun infodecomp LSF with the updated chunglu_triplets.py"
n_trip = trp[["marker_1", "marker_2", "marker_3"]].drop_duplicates().shape[0]
print(f"triplet rows: {len(trp):,}  ({trp['component'].nunique():,} cells, {n_trip} triplets)")

In [ ]:
# Doublets are large (~50M rows); keep only the constituent pairs of our triplets (polars stream).
needed = idu.needed_pair_set(trp)
a, b = pl.col("marker_1"), pl.col("marker_2")
pair_expr = (pl.when(a <= b)
               .then(pl.concat_str([a, pl.lit("/"), b]))
               .otherwise(pl.concat_str([b, pl.lit("/"), a])).alias("pair"))
dbl = (pl.scan_parquet(str(OUT / "*_doublets.parquet"))
         .select(["component", "marker_1", "marker_2", "J", "EW", "join_count_z"])
         .with_columns(pair_expr)
         .filter(pl.col("pair").is_in(list(needed)))
         .collect()
         .to_pandas())
print(f"doublet rows (needed pairs only): {len(dbl):,}  ({dbl['pair'].nunique()} pairs)")

In [ ]:
tab = idu.build_joined_table(trp, dbl, sets=sets_map)   # attaches z_AB/z_AC/z_BC + e_AB/e_AC/e_BC
tab = idu.add_pairwise_prediction(tab)                   # -> W_pair, connected_z, doublet_explained_fraction
tab = tab.merge(meta, left_on="component", right_index=True, how="left")
print("rows per set:", tab["set"].value_counts().to_dict())
tab[["marker_1", "marker_2", "marker_3", "triplet_z", "z_AB", "z_AC", "z_BC",
     "W_pair", "connected_z", "doublet_explained_fraction", "set"]].head(4)

In [ ]:
# Verification: per-center columns sum EXACTLY to the totals (the extended-schema invariant).
assert np.allclose(trp["EW_cA"] + trp["EW_cB"] + trp["EW_cC"], trp["EW"]), "EW center split != EW"
assert np.allclose(trp["W_cA"] + trp["W_cB"] + trp["W_cC"], trp["W"]), "W center split != W"
print("per-center sums OK (EW_c*, W_c* reconstruct EW, W)")

# Drop degenerate (near-always-zero) triplets so R^2 is not diluted by empty motifs.
tab["tkey"] = list(zip(tab["marker_1"], tab["marker_2"], tab["marker_3"]))
nz = tab.assign(nz=tab["W"] > 0).groupby("tkey")["nz"].mean()
pop = set(nz[nz >= 0.20].index)
tab_pop = tab[tab["tkey"].isin(pop)].copy()
for c in ["triplet_z", "z_AB", "z_AC", "z_BC", "connected_z"]:
    tab_pop[c + "_a"] = np.arcsinh(tab_pop[c] / 5)
n_sel = tab_pop[tab_pop["set"] == "selected"]["tkey"].nunique()
n_ran = tab_pop[tab_pop["set"] == "random"]["tkey"].nunique()
print(f"populated triplets (>=20% of cells): {len(pop)}  (selected {n_sel}, random {n_ran})")

## Step 1 — pairs → triplets (empirical variance explained)

Leave-cells-out (GroupKFold) CV R² of `triplet_z` on its three constituent pair z's — blocking on
`component` so within-cell correlation cannot leak. `within_r2` (per-cell demeaned) vs `between_r2`
(cell means) separates genuine motif-level redundancy from a "busy cell" lifting every motif.
Reported for **selected** and **random**, on raw and asinh-stabilised z. Headline: `1 − cv_r2` =
fraction of triplet-z variance not recoverable from the doublets.

In [ ]:
X3 = ["z_AB", "z_AC", "z_BC"]

def r2_report(df, y="triplet_z"):
    rows = []
    for s in ["selected", "random"]:
        d = df[df["set"] == s]
        cv = idu.grouped_cv_r2(d, y, X3)
        wb = idu.within_between_r2(d, y, X3)
        rows.append(dict(set=s, cv_r2=cv["cv_r2"], cv_spearman_r2=cv["cv_spearman_r2"],
                         within_r2=wb["within_r2"], between_r2=wb["between_r2"], n=cv["n"]))
    return pd.DataFrame(rows)

print("=== pairs -> triplet z : leave-cells-out CV R^2 ===")
print("raw z:");   display(r2_report(tab_pop, "triplet_z").round(3))
print("asinh z:"); display(r2_report(tab_pop, "triplet_z_a").round(3))
print("added value of triplets = 1 - cv_r2 (variance NOT recoverable from the 3 doublet z's)")

In [ ]:
# Naive single-number predictors (rank-based, tail-robust): how far does just "the strongest pair"
# or "the average pair" get you toward the triplet z?
tmp = tab_pop.copy()
tmp["max_pair"] = tmp[X3].max(1)
tmp["mean_pair"] = tmp[X3].mean(1)
for s in ["selected", "random"]:
    d = tmp[tmp["set"] == s]
    print(f"[{s:8s}] spearman_R^2  max_pair={idu._spearman_r2(d['max_pair'], d['triplet_z']):.3f}  "
          f"mean_pair={idu._spearman_r2(d['mean_pair'], d['triplet_z']):.3f}")

In [ ]:
# Stratified by cell type (redundancy can differ by biology).
strat = []
for (s, ct), d in tab_pop.groupby(["set", "cell_type_annot"], observed=True):
    if len(d) < 2000:
        continue
    strat.append(dict(set=s, cell_type=ct,
                      cv_r2=idu.grouped_cv_r2(d, "triplet_z", X3)["cv_r2"], n=len(d)))
display(pd.DataFrame(strat).sort_values(["set", "cv_r2"]).round(3))

In [ ]:
# HEAVY (model fit) -- run on a compute/GPU kernel, not the login node.
# Nonlinear (gradient-boosting) CV R^2; gap over the linear cv_r2 = interaction redundancy
# captured beyond additive pair effects.
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

def gbm_cv_r2(df, y="triplet_z", x=None, max_n=300_000, seed=0):
    x = x or X3
    d = df[[y, "component"] + x].dropna()
    if len(d) > max_n:
        rng = np.random.default_rng(seed)
        g = d["component"].unique(); rng.shuffle(g)
        sizes = d.groupby("component").size(); keep = []; c = 0
        for gg in g:
            keep.append(gg); c += int(sizes[gg])
            if c >= max_n:
                break
        d = d[d["component"].isin(keep)]
    Y = d[y].to_numpy(); Xm = d[x].to_numpy(); gr = d["component"].to_numpy()
    oof = np.full(len(Y), np.nan)
    for tr, te in GroupKFold(5).split(Xm, Y, gr):
        m = HistGradientBoostingRegressor(max_depth=3, max_iter=300).fit(Xm[tr], Y[tr])
        oof[te] = m.predict(Xm[te])
    return float(1 - ((Y - oof) ** 2).sum() / ((Y - Y.mean()) ** 2).sum())

for s in ["selected", "random"]:
    print(f"[{s:8s}] GBM cv_r2 = {gbm_cv_r2(tab_pop[tab_pop['set'] == s]):.3f}")

## Step 2 — pairs → triplets (mechanistic connected decomposition)

`W_pair = e_AB·e_AC·EW_cA + e_AB·e_BC·EW_cB + e_AC·e_BC·EW_cC` is the mean-field "no-3-way" wedge:
each wedge center's degree-null mean scaled by the observed enrichments of its two incident pairs
(`e_gX = J_gX / EW_pair`). Since `triplet_z` and `connected_z = (W − W_pair)/√Var` share the same
`√Var` scale, `triplet_z − connected_z = (W_pair − EW)/√Var` is exactly the pairwise-explainable part,
and `doublet_explained_fraction = (W_pair − EW)/(W − EW)`. `connected_z` is the irreducible 3-way z.

In [ ]:
sub = tab_pop[tab_pop["W"] > 0].copy()
for s in ["selected", "random"]:
    d = sub[sub["set"] == s]
    print(f"[{s:8s}] median|triplet_z|={d['triplet_z'].abs().median():.2f}  "
          f"median|connected_z|={d['connected_z'].abs().median():.2f}  "
          f"median doublet_explained_fraction={d['doublet_explained_fraction'].median():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, s in zip(axes, ["selected", "random"]):
    d = sub[sub["set"] == s]
    ax.hist(np.arcsinh(d["triplet_z"] / 5), bins=60, alpha=.5, color="#4c72b0", label="triplet_z")
    ax.hist(np.arcsinh(d["connected_z"] / 5), bins=60, alpha=.5, color="#c44e52", label="connected_z")
    ax.set_title(f"{s}: total vs connected"); ax.set_xlabel("asinh(z/5)")
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
# Per-triplet: which survive as genuinely 3-way after removing the pairwise part?
per_t = (sub.groupby(["set", "marker_1", "marker_2", "marker_3"], observed=True)
            .agg(triplet_z=("triplet_z", "mean"), connected_z=("connected_z", "mean"),
                 frac=("doublet_explained_fraction", "median"), n=("W", "size")).reset_index())
per_t["survive"] = per_t["connected_z"].abs() >= 0.5 * per_t["triplet_z"].abs()
print("fraction of triplets keeping >=50% of |z| after removing the pairwise part:")
print(per_t.groupby("set")["survive"].mean().round(3).to_dict())
g = per_t[per_t["set"] == "selected"].copy()
g["lab"] = [tlab(a, b, c) for a, b, c in zip(g["marker_1"], g["marker_2"], g["marker_3"])]
print("\ntop genuinely-3-way selected triplets (large |connected_z|):")
display(g.reindex(g["connected_z"].abs().sort_values(ascending=False).index)
         [["lab", "triplet_z", "connected_z", "frac", "n"]].head(10).round(2))

### Step 2 validation — the pairwise enrichment factorisation

Confirm the mean-field factorisation `W_pair = Σ_center (e·e)·EW_center` reproduces the wedge mean under
the **enriched Chung–Lu** null (degrees + observed pair enrichments, **no** 3-way structure;
`bipartite_chunglu_enriched_sample_edges`) within a few %, on a handful of populated (cell, triplet)
probes. That sampler is *soft* (Chung–Lu), so we validate against the **soft** per-center means (`r` at
the center); the fixed- vs soft-degree correction (`r−s`) used by the analysis `W_pair` is validated
separately in the main Chung–Lu notebook's edge-swap gate. **User-run** (loads pxl graphs).

In [ ]:
# VALIDATION (user-run; loads graphs). W_pair should equal the wedge mean under the pairwise-only
# enriched Chung-Lu null within a few %.
from pixelator import read_pna
import chunglu_triplets as ct

marker_names = json.loads((OUT / "marker_names.json").read_text())
K = len(marker_names); midx = {m: i for i, m in enumerate(marker_names)}
def pxl(s):
    return str(BASE / "results" / s / "layout" / "layout" / f"{s}.layout.pxl")

probe = (sub[(sub["set"] == "selected") & (sub["W"] > 50)]
           .assign(ee=lambda x: x["e_AB"] * x["e_AC"] * x["e_BC"])
           .sort_values("ee", ascending=False).head(3))

NS = 150; rows = []
for _, rr in probe.iterrows():
    comp, smp = rr["component"], rr["sample"]
    el = read_pna([pxl(smp)]).filter(components=[comp]).edgelist().to_df()
    edges, labels, side, n = ct.load_cell(el[["umi1", "umi2", "marker_1", "marker_2"]], midx)
    A, L, d, twoE, N = ct.build_cell(edges, labels, K); m = d.sum() / 2
    s1, r1, t1, s2, r2, t2 = ct.side_strengths(d, labels, side, K)
    J = (L.T @ (A @ L)).toarray()
    EWp = (np.outer(s1, s2) + np.outer(s2, s1)) / m; np.fill_diagonal(EWp, (s1 * s2) / m)
    e = np.ones((K, K)); nzp = EWp > 1e-9; e[nzp] = np.clip(J[nzp] / EWp[nzp], 0, None)
    aa, bb, cc = sorted(midx[rr[c]] for c in ["marker_1", "marker_2", "marker_3"])
    # SOFT per-center means (r at the center, not r-s) to match the soft enriched sampler.
    EcA = (r1[aa]*s2[bb]*s2[cc] + r2[aa]*s1[bb]*s1[cc]) / m**2
    EcB = (s2[aa]*r1[bb]*s2[cc] + s1[aa]*r2[bb]*s1[cc]) / m**2
    EcC = (s2[aa]*s2[bb]*r1[cc] + s1[aa]*s1[bb]*r2[cc]) / m**2
    W_pair_soft = e[aa, bb]*e[aa, cc]*EcA + e[aa, bb]*e[bb, cc]*EcB + e[aa, cc]*e[bb, cc]*EcC
    mc = np.array([ct.wedge_count(ct.bipartite_chunglu_enriched_sample_edges(d, side, labels, e, j),
                                  labels, K, (aa, bb, cc)) for j in range(NS)])
    rows.append(dict(triplet=tlab(rr["marker_1"], rr["marker_2"], rr["marker_3"]),
                     W_obs=rr["W"], W_pair_fixed=rr["W_pair"], W_pair_soft=W_pair_soft,
                     mc_mean=mc.mean(), ratio=mc.mean() / W_pair_soft if W_pair_soft else np.nan))
val = pd.DataFrame(rows); display(val.round(2))
print("PASS: soft W_pair ≈ enriched-null MC (within 8%)" if (val["ratio"].sub(1).abs() < 0.08).all()
      else "NOTE: raise NS, or the leading-order factorisation deviates for very strong enrichments.")

## Step 3 — counts → pairs

`counts → raw pair count J`: the null mean `EW` **is** the count/degree prediction, so `R²(J, EW)` is
trivially high. `counts → standardized pair z`: regress `join_count_z` on marker abundance — should be
small if the null is calibrated (the z is the abundance-orthogonal residual). (Uses the constituent
pairs already loaded — a diverse but triplet-anchored pair sample.)

In [ ]:
dd = dbl.copy()
raw_ab = adata.obsm["clr"] if "clr" in adata.obsm else adata.layers["arcsinh"]
if not isinstance(raw_ab, pd.DataFrame):
    raw_ab = pd.DataFrame(np.asarray(raw_ab), index=adata.obs_names, columns=adata.var_names)
dd = dd[dd["component"].isin(set(raw_ab.index))].copy()
row = {c: i for i, c in enumerate(raw_ab.index)}
col = {mk: i for i, mk in enumerate(raw_ab.columns)}
abv = raw_ab.to_numpy()
ri = dd["component"].map(row).to_numpy().astype(int)
dd["clr_A"] = abv[ri, dd["marker_1"].map(col).to_numpy().astype(int)]
dd["clr_B"] = abv[ri, dd["marker_2"].map(col).to_numpy().astype(int)]
dd["clr_prod"] = dd["clr_A"] * dd["clr_B"]
dd["logEW"] = np.log1p(dd["EW"])

r2_JE = idu.linear_r2(dd["J"].to_numpy(), dd[["EW"]].to_numpy())
print(f"counts -> raw pair count J    : R^2(J, EW) = {r2_JE:.3f}   "
      f"(EW is the pure count/degree prediction -> trivially high)")
feat = ["clr_A", "clr_B", "clr_prod", "logEW"]
rc = idu.grouped_cv_r2(dd, "join_count_z", feat)
print(f"counts -> standardized pair z : CV R^2 = {rc['cv_r2']:.3f}   "
      f"(residual abundance dependence -> small if the null is calibrated)")
print(f"pair-z calibration: mean={dd['join_count_z'].mean():.2f}  sd={dd['join_count_z'].std():.2f}")

## Step 4 — the information ladder

In [ ]:
# Marginal variance explained / redundancy at each rung (selected vs random).
def rung_vals(s):
    d = tab_pop[tab_pop["set"] == s]
    r_pt = idu.grouped_cv_r2(d, "triplet_z", X3)["cv_r2"]
    dm = d[d["W"] > 0]["doublet_explained_fraction"].clip(0, 1).median()
    return r_pt, dm

sel_pt, sel_mech = rung_vals("selected")
ran_pt, ran_mech = rung_vals("random")
rungs = ["counts→J", "counts→pair z", "pairs→triplet z\n(CV R²)", "pairs→triplet\nexcess (mech.)"]
vals_sel = [r2_JE, rc["cv_r2"], sel_pt, sel_mech]
vals_ran = [r2_JE, rc["cv_r2"], ran_pt, ran_mech]
x = np.arange(len(rungs)); w = 0.38
fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.bar(x - w/2, vals_sel, w, color="#9467bd", label="selected")
ax.bar(x + w/2, vals_ran, w, color="#2ca02c", label="random")
for xi, (vs, vr) in enumerate(zip(vals_sel, vals_ran)):
    ax.text(xi - w/2, vs + .01, f"{vs:.2f}", ha="center", fontsize=7)
    ax.text(xi + w/2, vr + .01, f"{vr:.2f}", ha="center", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(rungs, fontsize=8)
ax.set_ylim(0, 1.05); ax.set_ylabel("variance explained / share")
ax.set_title("Each rung largely restates the one below — the 3-way remainder is the new signal")
ax.legend(fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()
print("counts→J and counts→pair z are pair-level (set-independent); the same value is drawn under each set for scale.")

## Methods note

- **Statistics reused as-is** — per-cell bipartite Chung–Lu / fixed-degree wedge (`triplet_z`) and
  join-count (`join_count_z`) z-scores from `chunglu_triplets.py`. The module now also emits the
  **per-center** wedge means/counts (`EW_cA/cB/cC`, `W_cA/cB/cC`; verified to sum to `EW`/`W`), which
  enable the pairwise (connected) decomposition without re-loading graphs.
- **pairs → triplets, empirical** — `triplet_z ~ [z_AB, z_AC, z_BC]`, leave-cells-out (GroupKFold, block
  on `component`) CV R²; Pearson- and Spearman-R²; asinh(z/5)-stabilised variant; `within_r2`
  (per-cell demeaned) vs `between_r2` (cell means) to strip the busy-cell confound. Optional nonlinear
  (GBM) CV R² for interaction redundancy.
- **pairs → triplets, mechanistic** — mean-field pairwise wedge `W_pair` (each center scaled by its two
  incident pair enrichments `e = J/EW_pair`), `connected_z = (W − W_pair)/√Var`,
  `doublet_explained_fraction = (W_pair − EW)/(W − EW)`. Validated (±few %) against a pairwise-only
  **enriched Chung–Lu** generative null (`bipartite_chunglu_enriched_sample_edges`).
- **counts → pairs** — `R²(J, EW)` (trivially high; `EW` is the count/degree prediction) vs
  `CV R²(join_count_z ~ abundance)` (small if the null is calibrated). Abundance from `obsm['clr']`.
- **Rigor** — cells are the replication unit; all R² grouped-CV, no naive pooled p-values over the
  correlated motif rows; heavy tails handled with rank/asinh variants; **selected** (enriched, biased
  upward) reported beside a **random** baseline (unbiased added-value estimate). Triplets sharing a
  pair are non-independent (noted, not corrected). The counts→pairs sample is triplet-anchored, not a
  uniform pair sample.